In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

for tabela in [
    "tickets",
    "users",
    "tickets_users",
    "tickets_tickets",
    "ticketsatisfactions",
]:
    
    spark.table(f"{catalog}.{schema}.bronze_{tabela}") \
        .createOrReplaceTempView(f"glpi_{tabela}")

    print(f"Extraindo tabela {tabela}")

In [0]:
# --- silver_Tickets ---
silver_tickets = spark.sql(""" 
SELECT

    id                      AS ticket_id,
    name                    AS titulo,

    status                  AS status_id,

    CASE
        WHEN status = 6 THEN 'Chamado Fechado'
        WHEN status = 5 THEN 'Solucionado'
        WHEN status = 4 THEN 'Pendente'
        WHEN status = 3 THEN 'Em Atendimento (Planejado)'
        WHEN status = 2 THEN 'Em Atendimento (Atribuído)'
        WHEN status = 1 THEN 'Novo'
        ELSE 'Desconhecido'
    END                     AS status_descricao,

    priority,
    urgency,
    impact,
    content,
    itilcategories_id,
    entities_id,
    users_id_recipient      AS usuario_solicitante_id,

    date_creation           AS data_criacao,
    date                    AS data_abertura,
    solvedate               AS data_solucao,
    closedate               AS data_fechamento,
    date_mod                AS data_modificacao,

    CASE
        WHEN is_deleted = 1 THEN TRUE
        ELSE FALSE
    END                     AS deletado,

    timestampdiff(
        HOUR,
        date_creation,
        solvedate
    )                       AS horas_resolucao

FROM (

    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY id
               ORDER BY date_mod DESC
           ) AS rn

    FROM glpi_tickets
)

WHERE rn = 1
  AND lower(coalesce(name,'')) NOT LIKE '%teste%'                    
""")


# --- silver_Users ---
silver_users = spark.sql("""
SELECT

    id              AS user_id,
    name            AS login,
    realname,
    firstname,

    concat_ws(
        ' ',
        firstname,
        realname
    )               AS nome_completo

FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY id
               ORDER BY date_mod DESC
           ) AS rn
    FROM glpi_users
)
WHERE rn = 1
""")


# --- Ticket Users ---
silver_tickets_users = spark.sql("""
SELECT

    tickets_id      AS ticket_id,
    users_id        AS user_id,
    type

FROM glpi_tickets_users
""")


# --- Ticket Links ---    
silver_tickets_tickets = spark.sql("""
SELECT *
FROM glpi_tickets_tickets
""")


# --- Ticket Satisfactions ---
silver_ticketsatisfactions = spark.sql("""
SELECT *
FROM glpi_ticketsatisfactions
""")

def save_tabelas(df, tabela):
    df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_{tabela}")
    
for tabela in [
    "tickets",
    "users",
    "tickets_users",
    "tickets_tickets",
    "ticketsatisfactions",
]:
    save_tabelas(eval(f"silver_{tabela}"), tabela)
    print(f"silver_{tabela} criada")